In [1]:
# IMPORT LIBRARY
import os
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# SEED UNTUK REPRODUCIBILITY
def seed_everything(seed=2):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"

seed_everything(2)

In [3]:
# HIPERPARAMETER OPTIMAL
best_params = {
    "layers": 2,
    "units": 64,
    "learning_rate": 0.001,
    "minibatch_size": 256,
    "weight_decay": 0.0
}

In [4]:
# MODEL LSTM
def build_best_lstm(n_timesteps, n_features, params):
    model = Sequential()
    model.add(Input(shape=(n_timesteps, n_features)))

    for i in range(params["layers"]):
        return_seq = True if i < params["layers"] - 1 else False

        model.add(
            LSTM(
                units = params["units"],
                return_sequences = return_seq,
                dropout = 0.0,
                recurrent_dropout = 0.0 
            )
        )

    model.add(Dense(1))

    optimizer = Adam(
        learning_rate=params["learning_rate"]
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

In [5]:
# PENILAIAN HARGA TEORITIS OPSI PUT
results_summary = []

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.5f" % x)

for split in range(1, 13):
    seed_everything(2)

    data = np.load(f"lstm_dataset_split_{split}.npz")

    X_train = data["X_train"]
    Y_train = data["Y_train"]
    X_val   = data["X_val"]
    Y_val   = data["Y_val"]
    X_test  = data["X_test"]
    Y_test  = data["Y_test"]

    contract_test = data["contract_test"]
    date_test     = data["date_test"]

    n_timesteps = X_train.shape[1]
    n_features  = X_train.shape[2]

    model = build_best_lstm(n_timesteps, n_features, best_params)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=100,
        batch_size=best_params["minibatch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    best_epoch = np.argmin(history.history["val_loss"]) + 1

    best_train_rmse = history.history["rmse"][best_epoch - 1]
    best_val_rmse   = history.history["val_rmse"][best_epoch - 1]

    y_pred = model.predict(X_test, verbose=0).flatten()

    rmse_scaled = np.sqrt(mean_squared_error(Y_test, y_pred))
    mae_scaled  = mean_absolute_error(Y_test, y_pred)

    y_scaler = joblib.load(f"scaler_Y_split_{split}.pkl")

    Y_test_inv = y_scaler.inverse_transform(Y_test.reshape(-1, 1)).ravel()
    y_pred_inv = y_scaler.inverse_transform(y_pred.reshape(-1, 1)).ravel()

    y_pred_inv = np.maximum(y_pred_inv, 0)

    rmse = np.sqrt(mean_squared_error(Y_test_inv, y_pred_inv))
    mae  = mean_absolute_error(Y_test_inv, y_pred_inv)

    df_result = pd.DataFrame({
        "ID_OPTION": contract_test,
        "QUOTE_DATE": pd.to_datetime(date_test),
        "MID_PRICE": Y_test_inv,
        "LSTM_PRICE": y_pred_inv
    })

    joblib.dump(df_result, f"Penilaian_Model_LSTM_{split}.pkl")

    results_summary.append({
        "split": split,
        "best_epoch": best_epoch,
        "RMSE_Pelatihan": best_train_rmse,
        "RMSE_Validasi": best_val_rmse,
        "RMSE_Pengujian": rmse_scaled,
        "RMSE_Skala_Data_Asli": rmse,
    })

In [6]:
# RINGKASAN AKHIR
df_summary = pd.DataFrame(results_summary)

print("\n===== RINGKASAN HASIL =====")
print(df_summary)


===== RINGKASAN HASIL =====
    split  best_epoch  RMSE_Pelatihan  RMSE_Validasi  RMSE_Pengujian  \
0       1          16         0.00250        0.00209         0.00222   
1       2          16         0.00213        0.00190         0.00640   
2       3          11         0.00245        0.00344         0.00667   
3       4          13         0.00245        0.00414         0.00822   
4       5           6         0.00361        0.00414         0.01179   
5       6          12         0.00261        0.00592         0.01711   
6       7          10         0.00311        0.00475         0.00798   
7       8          27         0.00254        0.00333         0.00608   
8       9           5         0.00517        0.00592         0.00451   
9      10           9         0.00381        0.00380         0.01194   
10     11           4         0.00505        0.00944         0.01499   
11     12          51         0.00219        0.00346         0.00452   

    RMSE_Skala_Data_Asli  
0      